# 문서 파싱 — 다양한 포맷에서 텍스트 꺼내기

In [1]:
# 실습용 텍스트 파일 만들기 

# UTF-8 파일 생성
with open("faq.txt", "w", encoding="utf-8") as f:
    f.write("Q: 환불은 언제까지 가능한가요?\n")
    f.write("A: 구매일로부터 7일 이내에만 가능합니다.\n\n")
    f.write("Q: 배송은 얼마나 걸리나요?\n")
    f.write("A: 결제 완료 후 2~3일 이내에 배송됩니다.\n")

# EUC-KR 파일 생성
with open("old_doc.txt", "w", encoding="euc-kr") as f:
    f.write("제1조 본 약관은 고객의 권리와 의무를 규정합니다.\n")
    f.write("제2조 서비스 이용 시간은 평일 09시~18시입니다.\n")

In [2]:
# 1. UTF-8로 읽기 (최근 파일 대부분)
with open("faq.txt", encoding="utf-8") as f:
    text = f.read()
print(text[:100])

# 2. EUC-KR로 읽기 (오래된 한국 문서)
with open("old_doc.txt", encoding="euc-kr") as f:
    text = f.read()
print(text[:100])

# 3. 인코딩을 모를 때 — chardet 자동 감지
import chardet

with open("old_doc.txt", "rb") as f:
    raw = f.read()
    result = chardet.detect(raw)
    print(f"감지 결과: {result['encoding']}, 신뢰도: {result['confidence']}")

text = raw.decode(result["encoding"])
print(text[:100])

Q: 환불은 언제까지 가능한가요?
A: 구매일로부터 7일 이내에만 가능합니다.

Q: 배송은 얼마나 걸리나요?
A: 결제 완료 후 2~3일 이내에 배송됩니다.

제1조 본 약관은 고객의 권리와 의무를 규정합니다.
제2조 서비스 이용 시간은 평일 09시~18시입니다.

감지 결과: CP949, 신뢰도: 0.78679685615487
제1조 본 약관은 고객의 권리와 의무를 규정합니다.
제2조 서비스 이용 시간은 평일 09시~18시입니다.



## PDF 파싱 실습 — PyMuPDF

### sample.pdf는 사전에 업로드 후 다운받으라고 하기

In [3]:
# ── 실습용 sample.pdf 만들기 (3페이지) ──
# pip install fpdf2
from fpdf import FPDF
from PIL import Image, ImageDraw, ImageFont
import os

pdf = FPDF()
pdf.add_font("malgun", "", "C:/Windows/Fonts/malgun.ttf")

# ── 1페이지: 자주 묻는 질문 ──
pdf.add_page()
pdf.set_font("malgun", size=16)
pdf.cell(0, 12, "자주 묻는 질문 (FAQ)", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("malgun", size=12)
pdf.cell(0, 8, "", new_x="LMARGIN", new_y="NEXT")

faq_page1 = [
    ("Q: 환불은 언제까지 가능한가요?", "A: 구매일로부터 7일 이내에만 가능합니다."),
    ("Q: 배송은 얼마나 걸리나요?", "A: 결제 완료 후 2~3일 이내에 배송됩니다."),
    ("Q: 회원 탈퇴는 어떻게 하나요?", "A: 마이페이지 > 설정 > 회원탈퇴에서 가능합니다."),
]

for q, a in faq_page1:
    pdf.cell(0, 10, q, new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 10, a, new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, "", new_x="LMARGIN", new_y="NEXT")

# ── 2페이지: 이용 약관 ──
pdf.add_page()
pdf.set_font("malgun", size=16)
pdf.cell(0, 12, "서비스 이용 약관", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("malgun", size=12)
pdf.cell(0, 8, "", new_x="LMARGIN", new_y="NEXT")

terms = [
    "제1조 본 약관은 고객의 권리와 의무를 규정합니다.",
    "제2조 서비스 이용 시간은 평일 09시부터 18시까지입니다.",
    "제3조 개인정보는 수집 목적 달성 후 즉시 파기합니다.",
    "제4조 서비스 내용은 사전 공지 후 변경될 수 있습니다.",
    "제5조 고객센터 연락처: 1588-0000 (평일 09~18시)",
]

for t in terms:
    pdf.cell(0, 10, t, new_x="LMARGIN", new_y="NEXT")

# ── 3페이지: 스캔본 (이미지로 된 텍스트) ──
# 먼저 텍스트를 이미지로 만든다
img = Image.new("RGB", (800, 400), "white")
draw = ImageDraw.Draw(img)
font = ImageFont.truetype("C:/Windows/Fonts/malgun.ttf", 24)

scan_lines = [
    "[ 스캔 문서 ]",
    "",
    "반품 및 교환 정책 안내",
    "",
    "1. 상품 수령 후 7일 이내 반품 가능",
    "2. 포장을 개봉한 경우 교환만 가능",
    "3. 고객 변심에 의한 반품 시 배송비 부담",
    "4. 불량 상품은 무조건 전액 환불",
]

y = 30
for line in scan_lines:
    draw.text((40, y), line, fill="black", font=font)
    y += 40

img.save("scan_page.png")

# 이미지를 PDF 3페이지에 삽입
pdf.add_page()
pdf.image("scan_page.png", x=10, y=10, w=190)

pdf.output("sample.pdf")
os.remove("scan_page.png")  # 임시 이미지 삭제
print("sample.pdf 생성 완료! (3페이지)")# ── 실습용 sample.pdf 만들기 (3페이지) ──
# pip install fpdf2
from fpdf import FPDF
from PIL import Image, ImageDraw, ImageFont
import os

pdf = FPDF()
pdf.add_font("malgun", "", "C:/Windows/Fonts/malgun.ttf")

# ── 1페이지: 자주 묻는 질문 ──
pdf.add_page()
pdf.set_font("malgun", size=16)
pdf.cell(0, 12, "자주 묻는 질문 (FAQ)", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("malgun", size=12)
pdf.cell(0, 8, "", new_x="LMARGIN", new_y="NEXT")

faq_page1 = [
    ("Q: 환불은 언제까지 가능한가요?", "A: 구매일로부터 7일 이내에만 가능합니다."),
    ("Q: 배송은 얼마나 걸리나요?", "A: 결제 완료 후 2~3일 이내에 배송됩니다."),
    ("Q: 회원 탈퇴는 어떻게 하나요?", "A: 마이페이지 > 설정 > 회원탈퇴에서 가능합니다."),
]

for q, a in faq_page1:
    pdf.cell(0, 10, q, new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 10, a, new_x="LMARGIN", new_y="NEXT")
    pdf.cell(0, 5, "", new_x="LMARGIN", new_y="NEXT")

# ── 2페이지: 이용 약관 ──
pdf.add_page()
pdf.set_font("malgun", size=16)
pdf.cell(0, 12, "서비스 이용 약관", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("malgun", size=12)
pdf.cell(0, 8, "", new_x="LMARGIN", new_y="NEXT")

terms = [
    "제1조 본 약관은 고객의 권리와 의무를 규정합니다.",
    "제2조 서비스 이용 시간은 평일 09시부터 18시까지입니다.",
    "제3조 개인정보는 수집 목적 달성 후 즉시 파기합니다.",
    "제4조 서비스 내용은 사전 공지 후 변경될 수 있습니다.",
    "제5조 고객센터 연락처: 1588-0000 (평일 09~18시)",
]

for t in terms:
    pdf.cell(0, 10, t, new_x="LMARGIN", new_y="NEXT")

# ── 3페이지: 스캔본 (이미지로 된 텍스트) ──
# 먼저 텍스트를 이미지로 만든다
img = Image.new("RGB", (800, 400), "white")
draw = ImageDraw.Draw(img)
font = ImageFont.truetype("C:/Windows/Fonts/malgun.ttf", 24)

scan_lines = [
    "[ 스캔 문서 ]",
    "",
    "반품 및 교환 정책 안내",
    "",
    "1. 상품 수령 후 7일 이내 반품 가능",
    "2. 포장을 개봉한 경우 교환만 가능",
    "3. 고객 변심에 의한 반품 시 배송비 부담",
    "4. 불량 상품은 무조건 전액 환불",
]

y = 30
for line in scan_lines:
    draw.text((40, y), line, fill="black", font=font)
    y += 40

img.save("scan_page.png")

# 이미지를 PDF 3페이지에 삽입
pdf.add_page()
pdf.image("scan_page.png", x=10, y=10, w=190)

pdf.output("sample.pdf")
os.remove("scan_page.png")  # 임시 이미지 삭제
print("sample.pdf 생성 완료! (3페이지)")

MERG NOT subset; don't know how to subset; dropped
MERG NOT subset; don't know how to subset; dropped


sample.pdf 생성 완료! (3페이지)
sample.pdf 생성 완료! (3페이지)


In [4]:
import fitz  # pip install pymupdf

# PDF 열기
doc = fitz.open("sample.pdf")
print(f"총 페이지 수: {len(doc)}")

# 전체 텍스트 이어 붙이기
full_text = ""
for i, page in enumerate(doc):
    text = page.get_text()
    full_text += text
    print(f"--- {i+1}페이지 ({len(text)}자) ---")
    print(text[:200])  # 앞 200자만 미리보기
    print()

doc.close()
print(f"\n전체 추출 텍스트: {len(full_text)}자")

총 페이지 수: 3
--- 1페이지 (152자) ---
자주 묻는 질문 (FAQ)
Q: 환불은 언제까지 가능한가요?
A: 구매일로부터 7일 이내에만 가능합니다.
Q: 배송은 얼마나 걸리나요?
A: 결제 완료 후 2~3일 이내에 배송됩니다.
Q: 회원 탈퇴는 어떻게 하나요?
A: 마이페이지 > 설정 > 회원탈퇴에서 가능합니다.


--- 2페이지 (172자) ---
서비스 이용 약관
제1조 본 약관은 고객의 권리와 의무를 규정합니다.
제2조 서비스 이용 시간은 평일 09시부터 18시까지입니다.
제3조 개인정보는 수집 목적 달성 후 즉시 파기합니다.
제4조 서비스 내용은 사전 공지 후 변경될 수 있습니다.
제5조 고객센터 연락처: 1588-0000 (평일 09~18시)


--- 3페이지 (0자) ---



전체 추출 텍스트: 324자


## PDF 파싱의 한계와 OCR

In [5]:
import fitz
from PIL import Image
import easyocr
import numpy as np
import io

reader = easyocr.Reader(["ko", "en"])

doc = fitz.open("sample.pdf")

for i, page in enumerate(doc):
    text = page.get_text()

    if text.strip():
        print(f"--- {i+1}페이지 (텍스트 추출) ---")
        print(text[:200])
    else:
        print(f"--- {i+1}페이지 (텍스트 없음 → OCR 시도) ---")

        pix = page.get_pixmap(dpi=300)
        img = Image.open(io.BytesIO(pix.tobytes("png")))

        # ✅ numpy 배열로 변환해서 전달
        results = reader.readtext(np.array(img))
        ocr_text = "\n".join([r[1] for r in results])
        print(ocr_text)

doc.close()

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


--- 1페이지 (텍스트 추출) ---
자주 묻는 질문 (FAQ)
Q: 환불은 언제까지 가능한가요?
A: 구매일로부터 7일 이내에만 가능합니다.
Q: 배송은 얼마나 걸리나요?
A: 결제 완료 후 2~3일 이내에 배송됩니다.
Q: 회원 탈퇴는 어떻게 하나요?
A: 마이페이지 > 설정 > 회원탈퇴에서 가능합니다.

--- 2페이지 (텍스트 추출) ---
서비스 이용 약관
제1조 본 약관은 고객의 권리와 의무를 규정합니다.
제2조 서비스 이용 시간은 평일 09시부터 18시까지입니다.
제3조 개인정보는 수집 목적 달성 후 즉시 파기합니다.
제4조 서비스 내용은 사전 공지 후 변경될 수 있습니다.
제5조 고객센터 연락처: 1588-0000 (평일 09~18시)

--- 3페이지 (텍스트 없음 → OCR 시도) ---


d:\GitHub\1_lecture-materials\.venv311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


[스랜 문서 ]
반품 및 교환 정책 안내
1. 상품 수령 후 7일 이내 반품 가능
2. 포장올 개쁨한 경우 교환만 가능
3. 고객 변심에 의한 반품 시 배슷비 부담
4. 불량 상품은 무조건 전액 환불


# 고정 길이 + 오버랩 청킹

In [6]:
# ── 실습용 doc.txt 만들기 (고객센터 매뉴얼) ──

content = """고객센터 운영 매뉴얼

제1장 환불 정책
환불은 구매일로부터 7일 이내에만 가능합니다. 환불을 요청하려면 고객센터에 전화하거나 홈페이지의 환불 신청 페이지를 이용해 주세요. 환불 처리에는 영업일 기준 3~5일이 소요되며, 결제 수단에 따라 추가 시간이 걸릴 수 있습니다. 신용카드로 결제한 경우 카드사 정책에 따라 최대 7일이 추가로 소요될 수 있습니다. 현금 결제의 경우 등록된 계좌로 직접 환불됩니다.

제2장 배송 안내
주문 확인 후 1~2일 이내에 상품이 출고됩니다. 출고 이후 배송은 택배사 사정에 따라 1~3일이 소요됩니다. 제주도 및 도서산간 지역은 추가 배송일이 발생할 수 있습니다. 배송 추적은 마이페이지에서 확인 가능하며, 송장번호는 출고 완료 시 문자로 안내됩니다. 배송 중 파손이 발생한 경우 수령 후 24시간 이내에 고객센터로 연락해 주세요. 파손 사진을 함께 보내주시면 빠른 처리가 가능합니다.

제3장 회원 등급 제도
회원 등급은 최근 6개월간 구매 금액을 기준으로 매월 1일에 갱신됩니다. 등급은 일반, 실버, 골드, VIP 네 단계로 나뉩니다. 일반 등급은 가입 즉시 부여되며 별도의 혜택은 없습니다. 실버 등급은 6개월간 누적 구매 금액 10만원 이상일 때 부여되며, 전 상품 2% 적립 혜택이 제공됩니다. 골드 등급은 30만원 이상이며 5% 적립과 무료배송 혜택이 있습니다. VIP 등급은 50만원 이상이며 10% 적립, 무료배송, 전담 상담사 배정 혜택이 제공됩니다.

제4장 포인트 정책
포인트는 상품 구매 시 결제 금액의 1~10%가 적립됩니다. 적립률은 회원 등급에 따라 차등 적용됩니다. 포인트는 적립일로부터 1년간 유효하며, 유효기간이 지난 포인트는 자동 소멸됩니다. 포인트는 1,000포인트 이상부터 사용 가능하며, 결제 금액의 최대 30%까지 사용할 수 있습니다. 이벤트 적립 포인트는 별도의 유효기간이 적용될 수 있으니 마이페이지에서 확인해 주세요.

제5장 교환 및 반품
상품 수령 후 7일 이내에 교환 및 반품 신청이 가능합니다. 단, 고객의 단순 변심에 의한 반품의 경우 왕복 배송비 6,000원이 부과됩니다. 상품 하자로 인한 교환 및 반품은 배송비를 회사가 부담합니다. 교환은 동일 상품의 다른 옵션으로만 가능하며, 다른 상품으로의 교환은 반품 후 재주문해 주셔야 합니다. 반품 접수 후 상품 회수까지 2~3일, 환불 처리까지 추가 3~5일이 소요됩니다.
"""

with open("doc.txt", "w", encoding="utf-8") as f:
    f.write(content)

print(f"doc.txt 생성 완료! ({len(content)}자)")

doc.txt 생성 완료! (1184자)


In [7]:
def chunk_text_fixed_overlap(text: str, chunk_size: int = 300, overlap: int = 50):
    """고정 길이로 자르되, 인접 청크가 overlap만큼 겹치도록 한다."""
    assert 0 <= overlap < chunk_size, "overlap은 chunk_size보다 작아야 한다"

    chunks = []
    step = chunk_size - overlap   # 다음 청크의 시작 위치를 step만큼만 전진시킨다
    for start in range(0, len(text), step):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):   # 마지막 청크에 도달하면 종료
            break
    return chunks

# 사용 예시
text = open("doc.txt", encoding="utf-8").read()
chunks = chunk_text_fixed_overlap(text, chunk_size=300, overlap=50)
print(f"총 {len(chunks)}개 청크")
# print(chunks[0][:300], "...")
# print(chunks[1][:300], "...")
# print(chunks[2][:300], "...")
# print(chunks[3][:300], "...")
# print(chunks[4][:300], "...")
for i in range(len(chunks) - 1):
    tail = chunks[i][-50:]      # 현재 청크의 끝 50자
    head = chunks[i + 1][:50]   # 다음 청크의 시작 50자
    print(f"[청크{i}] ...끝 50자  : {tail!r}")
    print(f"[청크{i+1}] 시작 50자: {head!r}")
    print(f"일치 여부: {tail == head}")

총 5개 청크
[청크0] ...끝 50자  : ' 이내에 상품이 출고됩니다. 출고 이후 배송은 택배사 사정에 따라 1~3일이 소요됩니다. '
[청크1] 시작 50자: ' 이내에 상품이 출고됩니다. 출고 이후 배송은 택배사 사정에 따라 1~3일이 소요됩니다. '
일치 여부: True
[청크1] ...끝 50자  : '매월 1일에 갱신됩니다. 등급은 일반, 실버, 골드, VIP 네 단계로 나뉩니다. 일반 등'
[청크2] 시작 50자: '매월 1일에 갱신됩니다. 등급은 일반, 실버, 골드, VIP 네 단계로 나뉩니다. 일반 등'
일치 여부: True
[청크2] ...끝 50자  : '구매 시 결제 금액의 1~10%가 적립됩니다. 적립률은 회원 등급에 따라 차등 적용됩니다.'
[청크3] 시작 50자: '구매 시 결제 금액의 1~10%가 적립됩니다. 적립률은 회원 등급에 따라 차등 적용됩니다.'
일치 여부: True
[청크3] ...끝 50자  : ', 고객의 단순 변심에 의한 반품의 경우 왕복 배송비 6,000원이 부과됩니다. 상품 하자'
[청크4] 시작 50자: ', 고객의 단순 변심에 의한 반품의 경우 왕복 배송비 6,000원이 부과됩니다. 상품 하자'
일치 여부: True


# 문단 기반 청킹

In [8]:
def chunk_text_by_paragraph(text: str, max_size: int = 500):
    """문단(\n\n) 단위로 자르되, 너무 긴 문단은 고정 길이로 다시 자른다."""
    paragraphs = text.split("\n\n")
    chunks = []

    for p in paragraphs:
        p = p.strip()
        if not p:                       # 빈 문단은 건너뛴다
            continue
        if len(p) <= max_size:        # 한도 안이면 그대로 한 청크
            chunks.append(p)
        else:                          # 너무 길면 고정 길이로 재분할
            chunks.extend(
                chunk_text_fixed_overlap(p, chunk_size=max_size, overlap=50)
            )
    return chunks

# 사용 예시
text = open("doc.txt", encoding="utf-8").read()
chunks = chunk_text_by_paragraph(text, max_size=500)
print(f"총 {len(chunks)}개 청크")
for i, c in enumerate(chunks[:3]):
    print(f"[{i}] ({len(c)}자) {c[:60]}...")


총 6개 청크
[0] (11자) 고객센터 운영 매뉴얼...
[1] (213자) 제1장 환불 정책
환불은 구매일로부터 7일 이내에만 가능합니다. 환불을 요청하려면 고객센터에 전화하거나 홈페...
[2] (230자) 제2장 배송 안내
주문 확인 후 1~2일 이내에 상품이 출고됩니다. 출고 이후 배송은 택배사 사정에 따라 1...


# 청킹 결과 비교 — 같은 질문, 다른 결과

In [9]:
def score(chunk: str, query: str):
    """질문 단어가 청크에 등장한 횟수로 점수를 매긴다."""
    return sum(chunk.count(w) for w in query.split())

def top_k(chunks, query, k=3):
    """점수 높은 순으로 상위 k개 청크를 돌려준다."""
    ranked = sorted(chunks, key=lambda c: score(c, query), reverse=True)
    return ranked[:k]

# 두 전략으로 청킹
text = open("doc.txt", encoding="utf-8").read()
fixed_chunks = chunk_text_fixed_overlap(text, chunk_size=300, overlap=50)
para_chunks  = chunk_text_by_paragraph(text, max_size=500)

query = "환불 정책은 어떻게 되나요?"

# 같은 질문, 다른 결과
for name, chunks in [("고정+오버랩", fixed_chunks),
                     ("문단 기반"  , para_chunks)]:
    print(f"\n=== {name} (총 {len(chunks)}개) ===")
    for i, c in enumerate(top_k(chunks, query)):
        print(f"[Top{i+1}] score={score(c, query)} | {c[:60]}...")



=== 고정+오버랩 (총 5개) ===
[Top1] score=6 | 고객센터 운영 매뉴얼

제1장 환불 정책
환불은 구매일로부터 7일 이내에만 가능합니다. 환불을 요청하려면 고...
[Top2] score=1 | , 고객의 단순 변심에 의한 반품의 경우 왕복 배송비 6,000원이 부과됩니다. 상품 하자로 인한 교환 및 ...
[Top3] score=0 |  이내에 상품이 출고됩니다. 출고 이후 배송은 택배사 사정에 따라 1~3일이 소요됩니다. 제주도 및 도서산간...

=== 문단 기반 (총 6개) ===
[Top1] score=6 | 제1장 환불 정책
환불은 구매일로부터 7일 이내에만 가능합니다. 환불을 요청하려면 고객센터에 전화하거나 홈페...
[Top2] score=1 | 제5장 교환 및 반품
상품 수령 후 7일 이내에 교환 및 반품 신청이 가능합니다. 단, 고객의 단순 변심에 ...
[Top3] score=0 | 고객센터 운영 매뉴얼...


# ChromaDB로 벡터 저장소 구축

In [10]:
import chromadb

# 1) ./db 폴더에 영구 저장되는 클라이언트 생성
client = chromadb.PersistentClient(path="./db")
collection = client.get_or_create_collection(name="manual")

# 2) 앞에서 만든 함수로 문서를 청킹
text = open("doc.txt", encoding="utf-8").read()
chunks = chunk_text_by_paragraph(text, max_size=500)

# 3) 컬렉션에 추가 — 임베딩은 Chroma가 내부적으로 자동 생성
collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    metadatas=[{"source": "doc.txt", "idx": i} for i in range(len(chunks))],
)
print(f"저장된 청크 수: {collection.count()}")


저장된 청크 수: 6


# 전체 RAG 파이프라인 통합 구현

In [11]:
import fitz
import chromadb

def load_pdf(path: str) -> str:
    doc = fitz.open(path)
    return "\n\n".join(page.get_text() for page in doc)

# 1. 문서 로드 및 청킹
text = load_pdf("part3.pdf")
chunks = chunk_text_by_paragraph(text, max_size=500)
print(f"청크 수: {len(chunks)}")

# 2. 벡터 DB에 저장
client = chromadb.PersistentClient(path="./db")
collection = client.get_or_create_collection(name="rag_pipeline")
collection.upsert(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    metadatas=[{"source": "part3.pdf"}] * len(chunks)
)
print(f"저장된 청크 수: {collection.count()}")

청크 수: 7
저장된 청크 수: 7


In [12]:
from dotenv import load_dotenv
import os
import google.generativeai as genai

load_dotenv()  # .env 파일 로드
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

def ask_llm(prompt: str) -> str:
    model = genai.GenerativeModel("gemini-2.5-flash")
    return model.generate_content(prompt).text

# 1. 질문
query = "청킹 전략 방식 4가지는 무엇인가요?"

# 2. 유사 청크 검색 (Top-K)
results = collection.query(
    query_texts=[query],
    n_results=3
)
context = "\n\n".join(results["documents"][0])

# 3. LLM에 컨텍스트와 함께 질문
prompt = f"""다음 문서를 참고해 답하라.
검색 결과에 없는 내용은 모른다고 답하라.

[문서]
{context}

[질문] {query}
"""
answer = ask_llm(prompt)
print(answer)

d:\GitHub\1_lecture-materials\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\esdl\AppData\Local\Temp\ipykernel_4888\4214491450.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


문서에 따르면 청킹 전략 방식 4가지는 다음과 같습니다.

1.  **Fixed-size (고정 길이)**: 정해진 토큰·글자 수로 자르며, 보통 앞뒤가 겹치도록 오버랩을 둡니다.
2.  **Recursive (재귀적 분할)**: 문단·줄바꿈·문장 같은 구분자 우선순위를 따라 큰 단위부터 자르고, 한도를 넘으면 더 작은 단위로 다시 자릅니다.
3.  **Document-based (구조 기반)**: Markdown 헤더, HTML 태그, 코드 블록 등 문서 고유 구조를 경계로 삼습니다.
4.  **Semantic (의미 기반)**: 문장별 임베딩 유사도가 크게 떨어지는 지점을 의미 경계로 보고 자릅니다.


# 같은 내용인데 로컬로 진행

In [13]:
import fitz
import chromadb
import ollama

def load_pdf(path: str) -> str:
    doc = fitz.open(path)
    return "\n\n".join(page.get_text() for page in doc)

def ollama_embed(texts: list[str]) -> list[list[float]]:
    return [
        ollama.embed(model="nomic-embed-text", input=t)["embeddings"][0]
        for t in texts
    ]

# 1. 문서 로드 및 청킹
text = load_pdf("part3.pdf")
chunks = chunk_text_by_paragraph(text, max_size=500)
print(f"청크 수: {len(chunks)}")

# 2. 벡터 DB에 저장
client = chromadb.PersistentClient(path="./db")
collection = client.get_or_create_collection(name="rag_ollama")
collection.upsert(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=ollama_embed(chunks),
    metadatas=[{"source": "part3.pdf"}] * len(chunks)
)
print(f"저장된 청크 수: {collection.count()}")

청크 수: 7
저장된 청크 수: 7


In [14]:
import ollama

def ask_llm(prompt: str) -> str:
    response = ollama.chat(
        model="qwen2.5:3b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]

# 1. 질문
query = "청킹 전략 방식 4가지는 무엇인가요?"

# 2. 유사 청크 검색 (Top-K)
q_emb = ollama.embed(model="nomic-embed-text", input=query)["embeddings"][0]
results = collection.query(
    query_embeddings=[q_emb],
    n_results=3
)
context = "\n\n".join(results["documents"][0])

# 3. LLM에 컨텍스트와 함께 질문
prompt = f"""다음 문서를 참고해 답하라.
검색 결과에 없는 내용은 모른다고 답하라.

[문서]
{context}

[질문] {query}
"""
answer = ask_llm(prompt)
print(answer)

청킹의 4 가지 전략은 다음과 같습니다:

1. 고정 길이 분할: 특정 글자 수나 토큰 수를 기준으로 한 번에 자르는 방법입니다.
2. 오버랩 청킹: 이웃한 청크 사이에서 일부 내용이 겹치도록 잘라, 문장 중간에서 맥락이 끊기는 문제를 해결합니다.
3. 구조 기반 분할: 문서의 문단, 제목, 섹션 등 자연스러운 경계를 따라 자릅니다. 이렇게 하면 의미 단위가 보존됩니다.
4. 권장 크기 청킹: 일반적으로 300-800 토큰 크기를 권장하며, 오버랩 부분은 10-20% 수준에서 시작해 데이터에 맞게 조정합니다.

문서는 이 외에도 다양한 청킹 방법을 제시하고 있지만, 주요 청킹 방식은 이와 같습니다.
